# Fig 5E — TF motif accessibility vs RNA expression (examples)
Example TFs from the pseudotime trajectory: two with **rising** TF motif accessibility (ZEB1, GRHL2) and two **falling** (NFIC, ZBTB14). Each point is a pseudotime segment (colored NE root → differentiated tip, same palette as Fig S5D); x = TF motif accessibility coefficient (vs NE root), y = mean mRNA. GRHL2 is concordant (activity and expression rise together); ZEB1/NFIC/ZBTB14 are discordant. Full set in Fig S5H.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from scipy.stats import spearmanr
coef = load_matrix('PseudotimeTFActivity_coef.csv')
seg = sorted([c for c in coef.columns if c.startswith('PT_seg')], key=lambda c: int(c.replace('PT_seg','')))
coef = coef[seg]
mrna = load_matrix('mRNA_by_pseudotime_segment.csv'); mrna = mrna[[c for c in seg if c in mrna.columns]]
mrna = mrna[~mrna.index.duplicated(keep='first')]
MAIN = [('ZEB1','rise'), ('GRHL2','rise'), ('NFIC','fall'), ('ZBT14','fall')]  # coef rownames
segidx = np.arange(1, len(seg)+1); cmap = segment_cmap(len(seg))
fig, axs = plt.subplots(2, 2, figsize=(3.7, 3.5), layout='constrained')
axs = axs.ravel()
for ax, (t, d) in zip(axs, MAIN):
    g = tf_label(t); x = coef.loc[t].values.astype(float); y = mrna.loc[g].values.astype(float)
    rho = spearmanr(x, y)[0]
    sc = ax.scatter(x, y, c=segidx, cmap=cmap, vmin=0.5, vmax=len(seg)+0.5, s=13, edgecolor='k', linewidths=0.25)
    ax.set_title(f'{g}  ({d}, \u03c1={rho:.2f})', fontsize=6.5)
    ax.axvline(0, color='grey', lw=0.4, ls='--')
    ax.set_xlabel('TF motif accessibility (vs root)', fontsize=5.5); ax.tick_params(labelsize=5)
for ax in (axs[0], axs[2]): ax.set_ylabel('mean mRNA', fontsize=5.5)
cb = fig.colorbar(sc, ax=axs.tolist(), fraction=0.03, pad=0.02)
cb.set_label('pseudotime segment (1 = NE root)', fontsize=5.5); cb.ax.tick_params(labelsize=5)
savepanel(fig, 'Fig5E_ATACvsRNA_examples')
